In [1]:

"""
Moduł do obliczeń kredytowych:
  - harmonogram_annuitetowy(...)  -> raty równe
  - harmonogram_malejacy(...)     -> raty malejące
  - oblicz_rrso(...)              -> Rzeczywista Roczna Stopa Oprocentowania
 
Wymaga: pandas, numpy
"""
 
import pandas as pd
import numpy as np

    Wzór na ratę:
        ![alt text](image.png)
    gdzie:
        K - kwota kredytu
        r - miesięczna stopa procentowa = oprocentowanie_roczne / 12
        n - liczba rat
 
    Parametry
    ---------
    kwota : float                  kwota kredytu (np. 100000)
    oprocentowanie_roczne : float  np. 0.08 oznacza 8% rocznie
    liczba_rat : int               liczba rat miesięcznych
 
    Zwraca
    ------
    DataFrame z kolumnami: Nr raty, Rata, Kapitał, Odsetki, Kapitał pozostały

In [ ]:
def harmonogram_annuitetowy(kwota: float,
                            oprocentowanie_roczne: float,
                            liczba_rat: int) -> pd.DataFrame:
  
    r = oprocentowanie_roczne / 12
    n = liczba_rat
    K = kwota
 
    if r == 0:
        rata = K / n
    else:
        rata = K * (r * (1 + r) ** n) / ((1 + r) ** n - 1)
 
    rows = []
    pozostaly = K
    for i in range(1, n + 1):
        odsetki = pozostaly * r
        kapital = rata - odsetki
        pozostaly -= kapital
        rows.append({
            "Nr raty": i,
            "Rata": round(rata, 2),
            "Kapitał": round(kapital, 2),
            "Odsetki": round(odsetki, 2),
            "Kapitał pozostały": round(max(pozostaly, 0), 2),
        })
 
    return pd.DataFrame(rows)


    Harmonogram spłaty kredytu z ratą malejącą.
 
    Część kapitałowa jest stała: K/n
    Część odsetkowa maleje:      kapital_pozostaly * r
 
    Parametry – jak w funkcji annuitetowej.


In [3]:
def harmonogram_malejacy(kwota: float,
                         oprocentowanie_roczne: float,
                         liczba_rat: int) -> pd.DataFrame:

    r = oprocentowanie_roczne / 12
    n = liczba_rat
    K = kwota
    kapital_raty = K / n   # stała część kapitałowa
 
    rows = []
    pozostaly = K
    for i in range(1, n + 1):
        odsetki = pozostaly * r
        rata = kapital_raty + odsetki
        pozostaly -= kapital_raty
        rows.append({
            "Nr raty": i,
            "Rata": round(rata, 2),
            "Kapitał": round(kapital_raty, 2),
            "Odsetki": round(odsetki, 2),
            "Kapitał pozostały": round(max(pozostaly, 0), 2),
        })
 
    return pd.DataFrame(rows)


    """
    Oblicza RRSO zgodnie z dyrektywą 2008/48/WE (i ustawą o kredycie konsumenckim).
 
    Równanie bazowe:
        Σ Ck / (1+X)^tk  =  Σ Dl / (1+X)^sl
    gdzie:
        Ck – wypłaty kredytu w czasie tk (lat od momentu 0)
        Dl – spłaty (raty) w czasie sl (lat od momentu 0)
        X  – szukane RRSO (stopa roczna)
 
    Założenia tej implementacji:
      - jedna wypłata kredytu w momencie 0 = kwota_kredytu - prowizja - inne_koszty
        (klient dostaje "do ręki" mniej niż formalna kwota kredytu)
      - raty miesięczne; rata nr k płacona po k/12 lat
      - można też uznać, że klient dostaje pełną kwotę, a prowizję płaci osobno
        w t=0 – matematycznie to to samo (przepływ netto w t=0 = K - prowizja).
 
    Parametry
    ---------
    kwota_kredytu : float       formalna kwota kredytu (przed potrąceniem kosztów)
    harmonogram : DataFrame     wynik harmonogram_annuitetowy/malejacy
    prowizja : float            prowizja banku (zwykle pobierana z kwoty kredytu)
    inne_koszty : float         inne koszty wliczane do RRSO (ubezpieczenie itp.)
 
    Zwraca
    ------
    RRSO jako liczba (np. 0.0934 = 9.34%)
    """

In [ ]:
def oblicz_rrso(kwota_kredytu: float,
                harmonogram: pd.DataFrame,
                prowizja: float = 0.0,
                inne_koszty: float = 0.0) -> float:
    
    raty = harmonogram["Rata"].to_numpy()
    n = len(raty)
    # czasy w latach – rata k-ta (k=1..n) płacona po k/12 lat
    t = np.arange(1, n + 1) / 12.0
 
    # kwota faktycznie otrzymana przez kredytobiorcę w t=0
    netto_w_t0 = kwota_kredytu - prowizja - inne_koszty
 
    def npv(X: float) -> float:
        # wartość bieżąca rat minus to, co klient dostał
        return np.sum(raty / (1 + X) ** t) - netto_w_t0
 
    # Bisekcja na przedziale, w którym szukamy RRSO.
    # RRSO > -100%, w praktyce 0%–500% wystarczy aż nadto.
    lo, hi = -0.999, 10.0
    f_lo, f_hi = npv(lo), npv(hi)
    if f_lo * f_hi > 0:
        raise ValueError("Nie udało się znaleźć RRSO w przedziale (-99.9%, 1000%).")
    #bisekcja
    for _ in range(200):
        mid = (lo + hi) / 2
        f_mid = npv(mid)
        if abs(f_mid) < 1e-10 or (hi - lo) < 1e-12:
            return mid
        if f_lo * f_mid < 0:
            hi, f_hi = mid, f_mid
        else:
            lo, f_lo = mid, f_mid
    return (lo + hi) / 2

In [9]:
pd.options.display.float_format = "{:,.2f}".format
 
KWOTA = 100_000          # kredyt 100 000 zł
OPROC = 0.08             # 8% w skali roku
RATY  = 60               # 5 lat
PROWIZJA = 3000         # 3% prowizji wliczanej do RRSO
 
print("=" * 70)
print(f"Kredyt: {KWOTA:,.0f} zł | oproc. {OPROC*100:.2f}% | "
          f"{RATY} rat | prowizja {PROWIZJA:,.0f} zł")
print("=" * 70)
 

Kredyt: 100,000 zł | oproc. 8.00% | 60 rat | prowizja 3,000 zł


In [11]:
h_ann = harmonogram_annuitetowy(KWOTA, OPROC, RATY)
print("\n>>> RATA ANNUITETOWA (równa)")
print(h_ann.head(3).to_string(index=False))
print("   ...")
print(h_ann.tail(3).to_string(index=False))
print(f"\nSuma rat:     {h_ann['Rata'].sum():>12,.2f} zł")
print(f"Suma odsetek: {h_ann['Odsetki'].sum():>12,.2f} zł")
rrso_ann = oblicz_rrso(KWOTA, h_ann, prowizja=PROWIZJA)
print(f"RRSO:         {rrso_ann*100:>12,.4f} %")


>>> RATA ANNUITETOWA (równa)
 Nr raty     Rata  Kapitał  Odsetki  Kapitał pozostały
       1 2,027.64 1,360.97   666.67          98,639.03
       2 2,027.64 1,370.05   657.59          97,268.98
       3 2,027.64 1,379.18   648.46          95,889.80
   ...
 Nr raty     Rata  Kapitał  Odsetki  Kapitał pozostały
      58 2,027.64 1,987.62    40.02           4,015.08
      59 2,027.64 2,000.87    26.77           2,014.21
      60 2,027.64 2,014.21    13.43               0.00

Suma rat:       121,658.40 zł
Suma odsetek:    21,658.40 zł
RRSO:               9.7052 %


In [ ]:
 h_mal = harmonogram_malejacy(KWOTA, OPROC, RATY)
    print("\n>>> RATA MALEJĄCA")
    print(h_mal.head(3).to_string(index=False))
    print("   ...")
    print(h_mal.tail(3).to_string(index=False))
    print(f"\nSuma rat:     {h_mal['Rata'].sum():>12,.2f} zł")
    print(f"Suma odsetek: {h_mal['Odsetki'].sum():>12,.2f} zł")
    rrso_mal = oblicz_rrso(KWOTA, h_mal, prowizja=PROWIZJA)
    print(f"RRSO:         {rrso_mal*100:>12,.4f} %")